# 02 — Currency Analysis

**Purpose**: Analyse the impact of GBP/USD/EUR exchange rates on apparent wine price
performance, and how currency affects comparisons with USD-denominated equity indices.

All fine wine prices are in GBP. When a USD-based investor compares wine (GBP) to
the S&P 500 (USD), currency moves are a major source of divergence. This notebook
makes that effect explicit.

## Sections
1. Environment setup & data loading
2. FX data download (GBP/USD, GBP/EUR)
3. Cumulative Liv-ex returns: GBP vs USD vs EUR
4. Annual return decomposition: price vs FX
5. Crisis periods (GFC 2008, COVID 2020) — FX impact
6. Wine vs S&P 500 with and without FX adjustment
7. Brexit 2016 event study
8. GBP/USD historical context
9. Output summary & assertions

In [1]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import yfinance as yf

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
def _find_repo_root(start: Path) -> Path:
    for p in [start] + list(start.parents):
        if (p / ".git").exists():
            return p
    return start

REPO_ROOT  = _find_repo_root(Path.cwd())
DATA_DIR   = REPO_ROOT / "projects" / "correlation-diversification" / "data"
IMAGES_DIR = REPO_ROOT / "projects" / "correlation-diversification" / "images" / "currency"
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

LIVEX_PARQUET      = DATA_DIR / "livex_indices_clean.parquet"
COMPARISON_PARQUET = DATA_DIR / "comparison_assets_monthly.parquet"

print(f"REPO_ROOT  : {REPO_ROOT}")
print(f"DATA_DIR   : {DATA_DIR}")
print(f"IMAGES_DIR : {IMAGES_DIR}")

REPO_ROOT  : /home/runner/work/report-for-me/report-for-me
DATA_DIR   : /home/runner/work/report-for-me/report-for-me/projects/correlation-diversification/data
IMAGES_DIR : /home/runner/work/report-for-me/report-for-me/projects/correlation-diversification/images/currency


In [2]:
# ---------------------------------------------------------------------------
# Plot style
# ---------------------------------------------------------------------------
PALETTE = [
    "#9437ff", "#83D483", "#FFD166", "#F78C6B",
    "#4D87D0", "#EF476F", "#06D6A0", "#C23FB7", "#4A4A68",
]

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor":   "white",
    "axes.spines.top":  False,
    "axes.spines.right": False,
    "font.size":        11,
    "axes.titlesize":   13,
    "axes.labelsize":   11,
})


def save_fig(fig: plt.Figure, name: str) -> Path:
    """Save figure to images/currency/ and print confirmation."""
    path = IMAGES_DIR / name
    fig.savefig(path, dpi=150, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"Saved -> {path.relative_to(REPO_ROOT)}")
    return path


def yf_series(ticker: str, start: str = "2000-01-01") -> pd.Series:
    """Download a single ticker from yfinance and return Close as a 1-D Series.

    Compatible with both old yfinance API (returns Series) and new API
    (returns DataFrame with MultiIndex columns — we squeeze to 1-D).
    """
    raw = yf.download(ticker, start=start, auto_adjust=True, progress=False)["Close"]
    if isinstance(raw, pd.DataFrame):
        raw = raw.iloc[:, 0]  # new yfinance: (Close, Ticker) MultiIndex column
    raw.name = ticker
    return raw


print("Plot style configured.")

Plot style configured.


## 1. Load Shared Data

Load the parquet files produced by `01_data_setup.ipynb`.  
If they do not exist yet, we fall back gracefully to the raw CSVs and yfinance.

In [3]:
# ---------------------------------------------------------------------------
# Liv-ex index history
# ---------------------------------------------------------------------------
if LIVEX_PARQUET.exists():
    livex = pd.read_parquet(LIVEX_PARQUET)
    livex.index = pd.to_datetime(livex.index)
    print(f"Loaded livex from parquet: {livex.shape}")
else:
    print("livex_indices_clean.parquet not found — building from CSV")
    livex_raw = pd.read_csv(
        DATA_DIR / "liv-ex_index_history.csv",
        parse_dates=["date"],
        index_col="date",
        low_memory=False,
    )
    numeric_cols = livex_raw.select_dtypes(include="number").columns.tolist()
    livex = livex_raw[numeric_cols].copy()
    livex.index = pd.to_datetime(livex.index)
    livex = livex[livex.index >= "2000-01-01"].resample("ME").last()
    print(f"Built livex from CSV: {livex.shape}")

print(f"Columns: {list(livex.columns)}")
print(f"Date range: {livex.index.min().date()} -> {livex.index.max().date()}")

livex_indices_clean.parquet not found — building from CSV
Built livex from CSV: (314, 11)
Columns: ['Burgundy 150', 'California 50', 'Champagne 50', 'Italy 100', 'Liv-ex Bordeaux 500', 'Liv-ex Fine Wine 100', 'Liv-ex Fine Wine 1000', 'Liv-ex Fine Wine 50', 'Port 50', 'Rest of the World 60', 'Rhone 100']
Date range: 2000-01-31 -> 2026-02-28


In [4]:
# ---------------------------------------------------------------------------
# Comparison assets (S&P 500 TR, FTSE 100 TR, Gold)
# ---------------------------------------------------------------------------
if COMPARISON_PARQUET.exists():
    comp = pd.read_parquet(COMPARISON_PARQUET)
    comp.index = pd.to_datetime(comp.index)
    print(f"Loaded comparison assets from parquet: {comp.shape}")
else:
    print("comparison_assets_monthly.parquet not found — downloading from yfinance")
    tickers = {"^SP500TR": "sp500tr", "CUKX.L": "ftse", "GC=F": "gold"}
    frames = {}
    for ticker, col in tickers.items():
        daily = yf_series(ticker, start="2000-01-01")
        frames[col] = daily.resample("ME").last()
    comp = pd.DataFrame(frames)
    comp = comp[comp.index >= "2000-01-01"]
    print(f"Downloaded comparison assets: {comp.shape}")

print(f"Columns: {list(comp.columns)}")
print(f"Date range: {comp.index.min().date()} -> {comp.index.max().date()}")

comparison_assets_monthly.parquet not found — downloading from yfinance


Downloaded comparison assets: (315, 3)
Columns: ['sp500tr', 'ftse', 'gold']
Date range: 2000-01-31 -> 2026-03-31


## 2. Download FX Rates (GBP/USD and GBP/EUR)

`GBPUSD=X` and `GBPEUR=X` give the price of 1 GBP in USD / EUR respectively.  
A *rising* rate means GBP strengthens — wine looks cheaper to foreign buyers.  
A *falling* rate means GBP weakens — wine looks more expensive abroad in local currency terms.

In [5]:
FX_PARQUET = DATA_DIR / "fx_rates_monthly.parquet"

if FX_PARQUET.exists():
    fx = pd.read_parquet(FX_PARQUET)
    fx.index = pd.to_datetime(fx.index)
    print(f"Loaded FX from cache: {fx.shape}")
else:
    gbpusd_daily = yf_series("GBPUSD=X", start="2000-01-01")
    gbpeur_daily = yf_series("GBPEUR=X", start="2000-01-01")

    fx = pd.DataFrame({
        "gbpusd": gbpusd_daily.resample("ME").last(),
        "gbpeur": gbpeur_daily.resample("ME").last(),
    })
    fx = fx[fx.index >= "2000-01-01"]
    fx.to_parquet(FX_PARQUET)
    print(f"Downloaded and cached FX data: {fx.shape}")

print(f"Date range : {fx.index.min().date()} -> {fx.index.max().date()}")
print(f"GBP/USD    : {fx['gbpusd'].dropna().iloc[-1]:.4f}")
print(f"GBP/EUR    : {fx['gbpeur'].dropna().iloc[-1]:.4f}")

Loaded FX from cache: (271, 2)
Date range : 2003-09-30 -> 2026-03-31
GBP/USD    : 1.3332
GBP/EUR    : 1.1582


In [6]:
# ---------------------------------------------------------------------------
# Identify column names dynamically (names may vary slightly)
# ---------------------------------------------------------------------------
def find_col(df: pd.DataFrame, *keywords) -> str | None:
    """Return the first column whose lowercased name contains all keywords."""
    for col in df.columns:
        col_l = col.lower()
        if all(kw.lower() in col_l for kw in keywords):
            return col
    return None

lfw100_col  = find_col(livex, "fine", "100")
lfw1000_col = find_col(livex, "fine", "1000")
burg150_col = find_col(livex, "burgundy")

print(f"Liv-ex Fine Wine 100  : {lfw100_col}")
print(f"Liv-ex Fine Wine 1000 : {lfw1000_col}")
print(f"Burgundy 150          : {burg150_col}")

if lfw100_col is None:
    lfw100_col = livex.columns[0]
    print(f"Falling back to first column: {lfw100_col}")

# Common date range
start_date = max(
    livex[lfw100_col].dropna().index.min(),
    fx["gbpusd"].dropna().index.min(),
    pd.Timestamp("2000-01-01"),
)
print(f"\nAnalysis start date: {start_date.date()}")

Liv-ex Fine Wine 100  : Liv-ex Fine Wine 100
Liv-ex Fine Wine 1000 : Liv-ex Fine Wine 1000
Burgundy 150          : Burgundy 150

Analysis start date: 2003-12-31


In [7]:
# ---------------------------------------------------------------------------
# Build aligned master frame on a common monthly calendar
# ---------------------------------------------------------------------------
master_idx = pd.date_range(start=start_date, end=livex.index.max(), freq="ME")

wine_gbp = livex[lfw100_col].reindex(master_idx).ffill(limit=3)
gbpusd   = fx["gbpusd"].reindex(master_idx).ffill(limit=3)
gbpeur   = fx["gbpeur"].reindex(master_idx).ffill(limit=3)

# Convert wine index to USD and EUR
wine_usd = wine_gbp * gbpusd
wine_eur = wine_gbp * gbpeur

# S&P 500 TR
sp500_col = find_col(comp, "sp500") or find_col(comp, "sp") or comp.columns[0]
sp500 = comp[sp500_col].reindex(master_idx).ffill(limit=3)

print("Master frame built.")
print(f"wine_gbp NaN count : {wine_gbp.isna().sum()}")
print(f"gbpusd   NaN count : {gbpusd.isna().sum()}")
print(f"wine_usd NaN count : {wine_usd.isna().sum()}")

Master frame built.
wine_gbp NaN count : 0
gbpusd   NaN count : 0
wine_usd NaN count : 0


## 3. Cumulative Liv-ex Returns: GBP, USD, EUR

The same physical wine asset, viewed through three currency lenses.

In [8]:
def cumret(s: pd.Series, base: float = 100.0) -> pd.Series:
    """Rebase a price series to `base` at its first non-NaN value."""
    first = s.dropna().iloc[0]
    return (s / first) * base


fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(master_idx, cumret(wine_gbp), color=PALETTE[0], lw=2.0, label="Liv-ex Fine Wine 100 (GBP)")
ax.plot(master_idx, cumret(wine_usd), color=PALETTE[1], lw=2.0, label="Liv-ex Fine Wine 100 (USD)")
ax.plot(master_idx, cumret(wine_eur), color=PALETTE[2], lw=2.0, label="Liv-ex Fine Wine 100 (EUR)")

ax.set_xlabel("Date")
ax.set_ylabel("Rebased to 100")
ax.legend(fontsize=9, loc="upper left")
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f"))
ax.grid(axis="y", linewidth=0.4, alpha=0.6)

fig.text(
    0.5, -0.04,
    "Figure 1. Cumulative Liv-ex Fine Wine 100 returns rebased to 100, expressed in GBP (native currency),"
    " USD, and EUR.\nDivergence between the lines reflects sterling exchange-rate movements alone"
    " — the underlying GBP wine prices are identical in all three series.",
    ha="center", va="top", fontsize=9,
    bbox=dict(boxstyle="round,pad=0.3", facecolor="#f5f5f5", edgecolor="none"),
)

fig.tight_layout()
save_fig(fig, "01_cumulative_returns_gbp_usd_eur.png")

Saved -> projects/correlation-diversification/images/currency/01_cumulative_returns_gbp_usd_eur.png


PosixPath('/home/runner/work/report-for-me/report-for-me/projects/correlation-diversification/images/currency/01_cumulative_returns_gbp_usd_eur.png')

## 4. Annual Return Decomposition: Price vs FX Component

Total USD return = GBP price return * GBP/USD FX return (approximately).  
We decompose the two factors to show how much of the USD-denominated performance  
came from wine prices vs currency.

In [9]:
def annual_pct_change(s: pd.Series) -> pd.Series:
    """Annual returns from a monthly price series."""
    return s.resample("YE").last().pct_change().dropna()


wine_gbp_ann = annual_pct_change(wine_gbp)
gbpusd_ann   = annual_pct_change(gbpusd)
wine_usd_ann = annual_pct_change(wine_usd)

years = wine_gbp_ann.index.intersection(gbpusd_ann.index)
price_component = wine_gbp_ann.loc[years]
fx_component    = gbpusd_ann.loc[years]
total_usd       = wine_usd_ann.reindex(years)

year_labels = years.year.astype(str)
x = np.arange(len(year_labels))
width = 0.35

fig, ax = plt.subplots(figsize=(14, 6))

ax.bar(x - width / 2, price_component * 100, width,
       label="GBP Price Return", color=PALETTE[0], alpha=0.85)
ax.bar(x + width / 2, fx_component * 100, width,
       label="GBP/USD FX Return", color=PALETTE[2], alpha=0.85)
ax.plot(x, total_usd * 100, "o-", color=PALETTE[4], lw=1.5, ms=5,
        label="Total USD Return (approx.)")

ax.axhline(0, color="#333333", lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels(year_labels, rotation=45, ha="right")
ax.set_ylabel("Annual return (%)")
ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.grid(axis="y", linewidth=0.4, alpha=0.6)

fig.text(
    0.5, -0.09,
    "Figure 2. Annual Liv-ex Fine Wine 100 return decomposed into GBP price component (left bars)"
    " and GBP/USD FX component (right bars).\n"
    "The line shows the approximate total USD return. Sterling weakness amplifies USD returns;"
    " sterling strength reduces them.",
    ha="center", va="top", fontsize=9,
    bbox=dict(boxstyle="round,pad=0.3", facecolor="#f5f5f5", edgecolor="none"),
)

fig.tight_layout()
save_fig(fig, "02_annual_return_decomposition_price_vs_fx.png")

Saved -> projects/correlation-diversification/images/currency/02_annual_return_decomposition_price_vs_fx.png


PosixPath('/home/runner/work/report-for-me/report-for-me/projects/correlation-diversification/images/currency/02_annual_return_decomposition_price_vs_fx.png')

## 5. Crisis Periods: FX Impact During GFC 2008 and COVID 2020

Zoom in on the two major stress periods to show how currency affected the apparent  
wine drawdown for non-GBP investors.

In [10]:
def plot_crisis_window(
    ax: plt.Axes,
    start: str,
    end: str,
    title: str,
    panel_label: str,
) -> None:
    """Draw rebased wine series (GBP/USD/EUR) over a crisis window."""
    mask = (master_idx >= start) & (master_idx <= end)
    idx = master_idx[mask]

    w_gbp = cumret(wine_gbp[mask].reset_index(drop=True))
    w_usd = cumret(wine_usd[mask].reset_index(drop=True))
    w_eur = cumret(wine_eur[mask].reset_index(drop=True))
    fx_u  = cumret(gbpusd[mask].reset_index(drop=True))

    ax.plot(idx, w_gbp.values, color=PALETTE[0], lw=2,         label="Wine (GBP)")
    ax.plot(idx, w_usd.values, color=PALETTE[1], lw=2,         label="Wine (USD)")
    ax.plot(idx, w_eur.values, color=PALETTE[2], lw=2,         label="Wine (EUR)")
    ax.plot(idx, fx_u.values,  color=PALETTE[8], lw=1.5, ls="--", label="GBP/USD rate")

    ax.axhline(100, color="#cccccc", lw=0.8)
    ax.set_title(title)
    ax.set_ylabel("Rebased to 100")
    ax.legend(fontsize=8)
    ax.grid(axis="y", linewidth=0.4, alpha=0.6)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f"))
    ax.text(0.02, 0.04, panel_label, transform=ax.transAxes, fontsize=9, color="#555555")


fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=False)

plot_crisis_window(axes[0], "2007-06-30", "2010-12-31", "GFC 2008-2009", "(A)")
plot_crisis_window(axes[1], "2019-10-31", "2021-12-31", "COVID 2020",    "(B)")

fig.text(
    0.5, -0.06,
    "Figure 3. Liv-ex Fine Wine 100 during two crisis periods, rebased to 100 at the window start."
    " GBP/USD overlaid (dashed) shows the FX contribution.\n"
    "(A) GFC 2008-2009: sterling fell sharply, amplifying the GBP price decline for USD investors."
    " (B) COVID 2020: sterling initially weakened then recovered as markets stabilised.",
    ha="center", va="top", fontsize=9,
    bbox=dict(boxstyle="round,pad=0.3", facecolor="#f5f5f5", edgecolor="none"),
)

fig.tight_layout()
save_fig(fig, "03_crisis_periods_gfc_covid_fx_impact.png")

Saved -> projects/correlation-diversification/images/currency/03_crisis_periods_gfc_covid_fx_impact.png


PosixPath('/home/runner/work/report-for-me/report-for-me/projects/correlation-diversification/images/currency/03_crisis_periods_gfc_covid_fx_impact.png')

## 6. Wine vs S&P 500: With and Without FX Adjustment

For a USD-based investor, fine wine (priced in GBP) vs the S&P 500 (USD)  
involves an unavoidable currency translation. We show:
- Wine in GBP — pure price performance
- Wine in USD — GBP price + GBP/USD FX move
- S&P 500 TR — no currency friction

In [11]:
common_start = max(
    wine_gbp.dropna().index.min(),
    wine_usd.dropna().index.min(),
    sp500.dropna().index.min(),
)
mask_all = master_idx >= common_start

w_gbp_r = cumret(wine_gbp[mask_all])
w_usd_r = cumret(wine_usd[mask_all])
sp_r    = cumret(sp500[mask_all])
fx_r    = cumret(gbpusd[mask_all])

fig, axes = plt.subplots(2, 1, figsize=(12, 10), gridspec_kw={"height_ratios": [3, 1]})

ax = axes[0]
ax.plot(w_gbp_r.index, w_gbp_r.values, color=PALETTE[0], lw=2, label="Liv-ex 100 (GBP)")
ax.plot(w_usd_r.index, w_usd_r.values, color=PALETTE[1], lw=2, label="Liv-ex 100 (USD-adjusted)")
ax.plot(sp_r.index,    sp_r.values,    color=PALETTE[4], lw=2, label="S&P 500 TR (USD)")
ax.set_ylabel("Rebased to 100")
ax.legend(fontsize=9)
ax.grid(axis="y", linewidth=0.4, alpha=0.6)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f"))
ax.set_title("Wine vs S&P 500: GBP/USD Currency Adjustment")

ax2 = axes[1]
ax2.plot(fx_r.index, fx_r.values, color=PALETTE[8], lw=1.5, ls="--", label="GBP/USD (rebased)")
ax2.axhline(100, color="#cccccc", lw=0.8)
ax2.set_ylabel("GBP/USD rebased")
ax2.set_xlabel("Date")
ax2.legend(fontsize=8)
ax2.grid(axis="y", linewidth=0.4, alpha=0.6)
ax2.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f"))

fig.text(
    0.5, -0.03,
    "Figure 4. Top: Cumulative returns for Liv-ex Fine Wine 100 in GBP and USD-adjusted vs S&P 500 TR (USD)."
    "\nBottom: GBP/USD exchange rate rebased to 100. The gap between the two wine lines"
    " is entirely attributable to sterling movements.",
    ha="center", va="top", fontsize=9,
    bbox=dict(boxstyle="round,pad=0.3", facecolor="#f5f5f5", edgecolor="none"),
)

fig.tight_layout()
save_fig(fig, "04_wine_vs_sp500_fx_adjusted.png")

Saved -> projects/correlation-diversification/images/currency/04_wine_vs_sp500_fx_adjusted.png


PosixPath('/home/runner/work/report-for-me/report-for-me/projects/correlation-diversification/images/currency/04_wine_vs_sp500_fx_adjusted.png')

## 7. Brexit 2016 Event Study

Brexit vote: 23 June 2016. Sterling fell ~10% vs USD/EUR in the days after the vote.  
For foreign investors holding GBP-priced wine, this was an immediate currency loss.  
For GBP investors, the wine price itself barely moved.  

We show a ~18-month window rebased to 100 at the vote date.

In [12]:
BREXIT_DATE  = pd.Timestamp("2016-06-23")
WINDOW_START = "2015-01-31"
WINDOW_END   = "2018-06-30"

mask_br = (master_idx >= WINDOW_START) & (master_idx <= WINDOW_END)
idx_br  = master_idx[mask_br]

# Boolean numpy array: True for months on or before the Brexit vote
pre_mask = (idx_br <= BREXIT_DATE)  # already a numpy bool array


def rebase_at_event(s: pd.Series, pre: np.ndarray) -> pd.Series:
    """Rebase series to 100 at the last pre-event observation.

    `pre` must be a numpy bool array with the same length as `s`.
    """
    ref_vals = s.loc[pre]
    ref = ref_vals.dropna().iloc[-1] if not ref_vals.dropna().empty else s.dropna().iloc[0]
    return (s / ref) * 100


w_gbp_br  = rebase_at_event(wine_gbp[mask_br].reset_index(drop=True),  pre_mask)
w_usd_br  = rebase_at_event(wine_usd[mask_br].reset_index(drop=True),  pre_mask)
w_eur_br  = rebase_at_event(wine_eur[mask_br].reset_index(drop=True),  pre_mask)
sp_br     = rebase_at_event(sp500[mask_br].reset_index(drop=True),     pre_mask)
fx_usd_br = rebase_at_event(gbpusd[mask_br].reset_index(drop=True),    pre_mask)
fx_eur_br = rebase_at_event(
    fx["gbpeur"].reindex(master_idx).ffill(limit=3)[mask_br].reset_index(drop=True),
    pre_mask,
)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), gridspec_kw={"height_ratios": [3, 1]})

ax1.axvline(BREXIT_DATE, color="#888888", lw=1.5, ls=":", label="Brexit vote (23 Jun 2016)")
ax1.plot(idx_br, w_gbp_br.values, color=PALETTE[0], lw=2, label="Liv-ex 100 (GBP)")
ax1.plot(idx_br, w_usd_br.values, color=PALETTE[1], lw=2, label="Liv-ex 100 (USD-adjusted)")
ax1.plot(idx_br, w_eur_br.values, color=PALETTE[2], lw=2, label="Liv-ex 100 (EUR-adjusted)")
ax1.plot(idx_br, sp_br.values,    color=PALETTE[4], lw=2, label="S&P 500 TR (USD)")
ax1.axhline(100, color="#cccccc", lw=0.8)
ax1.set_ylabel("Rebased to 100 at Brexit vote")
ax1.legend(fontsize=9)
ax1.grid(axis="y", linewidth=0.4, alpha=0.6)
ax1.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f"))
ax1.set_title("Brexit 2016 Event Study: Wine vs Equities")

ax2.axvline(BREXIT_DATE, color="#888888", lw=1.5, ls=":")
ax2.plot(idx_br, fx_usd_br.values, color=PALETTE[3], lw=1.5, label="GBP/USD (rebased)")
ax2.plot(idx_br, fx_eur_br.values, color=PALETTE[5], lw=1.5, label="GBP/EUR (rebased)")
ax2.axhline(100, color="#cccccc", lw=0.8)
ax2.set_ylabel("FX rate rebased")
ax2.set_xlabel("Date")
ax2.legend(fontsize=8)
ax2.grid(axis="y", linewidth=0.4, alpha=0.6)
ax2.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f"))

fig.text(
    0.5, -0.04,
    "Figure 5. Brexit 2016 event study, rebased to 100 at the vote date."
    " Top: Liv-ex Fine Wine 100 in three currencies vs S&P 500 TR.\n"
    "Bottom: GBP/USD and GBP/EUR. Post-Brexit sterling decline created divergence between"
    " GBP and foreign-currency wine performance — GBP price was stable, USD/EUR values fell.",
    ha="center", va="top", fontsize=9,
    bbox=dict(boxstyle="round,pad=0.3", facecolor="#f5f5f5", edgecolor="none"),
)

fig.tight_layout()
save_fig(fig, "05_brexit_event_study.png")

Saved -> projects/correlation-diversification/images/currency/05_brexit_event_study.png


PosixPath('/home/runner/work/report-for-me/report-for-me/projects/correlation-diversification/images/currency/05_brexit_event_study.png')

## 8. GBP/USD Historical Context

Full GBP/USD and GBP/EUR history with key macro events annotated,  
providing the backdrop for all currency analysis above.

In [13]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(gbpusd.index, gbpusd.values, color=PALETTE[0], lw=2, label="GBP/USD")
ax.plot(gbpeur.index, gbpeur.values, color=PALETTE[2], lw=2, label="GBP/EUR")

# Annotate key events with arrows
events = {
    "GFC low\n(Jan 2009)": "2009-01-31",
    "Brexit vote\n(Jun 2016)": "2016-06-30",
    "COVID\n(Mar 2020)": "2020-03-31",
    "Truss budget\n(Sep 2022)": "2022-09-30",
}
nearby_usd = gbpusd.dropna()
for label, dt in events.items():
    ts = pd.Timestamp(dt)
    nearest_ts = nearby_usd.index[np.argmin(np.abs((nearby_usd.index - ts).days))]
    val = float(nearby_usd.loc[nearest_ts])
    ax.annotate(
        label,
        xy=(nearest_ts, val),
        xytext=(nearest_ts, val + 0.14),
        fontsize=7.5, ha="center", color="#444444",
        arrowprops=dict(arrowstyle="->", color="#888888", lw=0.8),
    )

ax.set_xlabel("Date")
ax.set_ylabel("Rate (1 GBP = x foreign currency)")
ax.legend(fontsize=9)
ax.grid(axis="y", linewidth=0.4, alpha=0.6)

fig.text(
    0.5, -0.07,
    "Figure 6. GBP/USD and GBP/EUR monthly exchange rates, 2000 to present, with key sterling events annotated.\n"
    "A declining rate means sterling weakening: GBP-denominated assets become cheaper for foreign buyers"
    " but returns are reduced when converted back to USD or EUR.",
    ha="center", va="top", fontsize=9,
    bbox=dict(boxstyle="round,pad=0.3", facecolor="#f5f5f5", edgecolor="none"),
)

fig.tight_layout()
save_fig(fig, "06_gbp_usd_eur_historical.png")

Saved -> projects/correlation-diversification/images/currency/06_gbp_usd_eur_historical.png


PosixPath('/home/runner/work/report-for-me/report-for-me/projects/correlation-diversification/images/currency/06_gbp_usd_eur_historical.png')

## 9. Output Summary & Assertions

In [14]:
EXPECTED_PNGS = [
    "01_cumulative_returns_gbp_usd_eur.png",
    "02_annual_return_decomposition_price_vs_fx.png",
    "03_crisis_periods_gfc_covid_fx_impact.png",
    "04_wine_vs_sp500_fx_adjusted.png",
    "05_brexit_event_study.png",
    "06_gbp_usd_eur_historical.png",
]

errors = []
print("=== Output file check ===")
for fname in EXPECTED_PNGS:
    path = IMAGES_DIR / fname
    if path.exists() and path.stat().st_size > 10_000:
        print(f"  OK   {fname}  ({path.stat().st_size // 1024} KB)")
    elif path.exists():
        errors.append(f"{fname} suspiciously small ({path.stat().st_size} bytes)")
        print(f"  WARN {fname}  ({path.stat().st_size} bytes)")
    else:
        errors.append(f"{fname} not found")
        print(f"  MISS {fname}")

print()
if errors:
    raise AssertionError("Missing or empty output files:\n" + "\n".join(errors))
print(f"All {len(EXPECTED_PNGS)} PNG files present and non-empty. Currency analysis complete.")

=== Output file check ===
  OK   01_cumulative_returns_gbp_usd_eur.png  (183 KB)
  OK   02_annual_return_decomposition_price_vs_fx.png  (142 KB)
  OK   03_crisis_periods_gfc_covid_fx_impact.png  (232 KB)
  OK   04_wine_vs_sp500_fx_adjusted.png  (238 KB)
  OK   05_brexit_event_study.png  (287 KB)
  OK   06_gbp_usd_eur_historical.png  (158 KB)

All 6 PNG files present and non-empty. Currency analysis complete.
